In [10]:
# Configure and run GPR seagrass carbon-stock map prediction.
# Paths are relative to codebase/v2/src.
project_path <- Sys.getenv("SEAGRASS_V2_ROOT", "")
model_path <- file.path("data", "review", "GPR_final.rds")
meadow_path <- file.path("data", "seagrass_eov_poly_2025.csv")
output_dir <- file.path("output", "pixel_grouped", "stocks")
depth_cm <- 100

kernel_cwd <- getwd()
if (!is.character(project_path) || length(project_path) != 1L || is.na(project_path)) {
  stop("project_path must be a single directory path.", call. = FALSE)
}
start_dirs <- if (nzchar(project_path)) path.expand(project_path) else {
  c(kernel_cwd, file.path(kernel_cwd, "Seagrass_carbon_stock_model"))
}
offsets <- c("", "src", file.path("codebase", "v2", "src"))
checked <- character()
bootstrap_path <- NA_character_
for (start in start_dirs) {
  directory <- normalizePath(start, winslash = "/", mustWork = FALSE)
  for (i in seq_len(40L)) {
    for (offset in offsets) {
      candidate <- file.path(directory, offset)
      checked <- c(checked, candidate)
      markers <- file.path(candidate, c(
        "predict_gpr_at_maps.R",
        file.path("modelling", "R", "init_repo.R"),
        file.path("modelling", "R", "project_root.R")
      ))
      if (all(file.exists(markers))) {
        bootstrap_path <- markers[[3L]]
        break
      }
    }
    if (!is.na(bootstrap_path) || nzchar(project_path)) break
    parent <- dirname(directory)
    if (identical(parent, directory)) break
    directory <- parent
  }
  if (!is.na(bootstrap_path)) break
}
if (is.na(bootstrap_path)) {
  stop(
    if (nzchar(project_path)) paste0("Invalid project_path: ", project_path) else
      "Cannot find the v2 project root.",
    "\nKernel working directory: ", kernel_cwd,
    "\nSet project_path (or SEAGRASS_V2_ROOT) to the clone root, codebase/v2, or codebase/v2/src.",
    "\nChecked:\n  ", paste(unique(checked), collapse = "\n  "),
    call. = FALSE
  )
}



In [11]:
sys.source(bootstrap_path, envir = .GlobalEnv)
project_root <- seagrass_find_v2_root(kernel_cwd, project_path = project_path)
script_path <- file.path(project_root, "predict_gpr_at_points.R")

cat("v2 script:", script_path, "\n")
local({
  previous_root <- Sys.getenv("SEAGRASS_V2_ROOT", unset = NA_character_)
  Sys.setenv(SEAGRASS_V2_ROOT = project_root)
  
  tryCatch(source(script_path, local = .GlobalEnv), finally = {
    if (is.na(previous_root)) Sys.unsetenv("SEAGRASS_V2_ROOT") else
      Sys.setenv(SEAGRASS_V2_ROOT = previous_root)
  })
})


v2 script: /home/jovyan/Seagrass_carbon_stock_model/codebase/v2/src/predict_gpr_at_points.R 
[1] "Predictor variables for raster covariates:"
 [1] "surf_fgco2_mean_molc_m2_yr"                             
 [2] "s3a_olci_errnt.20160425_20241231.l3m.cu.iop.adg_443.4km"
 [3] "vo_mean_1.5m_m_s"                                       
 [4] "uo_p90_1.5m_m_s"                                        
 [5] "vo_p90_1.5m_m_s"                                        
 [6] "sw_ke_mean_cm2_s2"                                      
 [7] "daily_std_wave_height_mean_m"                           
 [8] "fe_mean_monthly_1.5m_mmol_m3"                           
 [9] "bottomt_p95_daily_c"                                    
[10] "spco2_monthly_mean_pa"                                  
[11] "s3a_olci_errnt.20160425_20241231.l3m.cu.chl.chlor_a.4km"
[12] "s3a_olci_errnt.20160425_20241231.l3m.cu.rrs.rrs_620.4km"
[13] "no_sig_wave_heights_mean"                               
[14] "seagrass_species"                

In [12]:
sys.source(bootstrap_path, envir = .GlobalEnv)
project_root <- seagrass_find_v2_root(kernel_cwd, project_path = project_path)
script_path <- file.path(project_root, "predict_gpr_at_maps.R")

cat("v2 script:", script_path, "\n")
# local({
#   previous_root <- Sys.getenv("SEAGRASS_V2_ROOT", unset = NA_character_)
#   Sys.setenv(SEAGRASS_V2_ROOT = project_root)
  
#   tryCatch(source(script_path, local = .GlobalEnv), finally = {
#     if (is.na(previous_root)) Sys.unsetenv("SEAGRASS_V2_ROOT") else
#       Sys.setenv(SEAGRASS_V2_ROOT = previous_root)
#   })
# })
setwd(project_root)
source(script_path, local = .GlobalEnv)
prediction_args <- c(model_path, meadow_path, output_dir, as.character(depth_cm))
predicted_meadows <- predict_gpr_at_maps(prediction_args)


v2 script: /home/jovyan/Seagrass_carbon_stock_model/codebase/v2/src/predict_gpr_at_maps.R 
- One or more packages recorded in the lockfile are not installed.
- Use `renv::status()` for more details.


ERROR: Error: Missing required package(s): sf
Use renv::restore() for reproducible setup.
Or install explicitly: install.packages(c("sf")).
To allow runtime installs, set SEAGRASS_AUTO_INSTALL_PKGS=true.


In [ ]:
if (!exists("predicted_meadows")) {
  stop("Run the prediction cell first.", call. = FALSE)
}
print(table(predicted_meadows$prediction_status))
cat("Map and prediction tables saved under:", output_dir, "\n")
